In [ ]:
    ############    #############   Lifespan and API versioning   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.2 Backend Engineering
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Lifespan Events   #############   ##############   

 =>  The lifespan async context manager is the current, recommended way to run
       startup/shutdown logic in FastAPI -- everything before 'yield' runs on startup,
       everything after runs on shutdown, guaranteed.

 =>  It replaces the older '@app.on_event("startup")' / '@app.on_event("shutdown")'
       decorators, which are deprecated -- lifespan composes better and correctly shares
       state (e.g. a DB pool) between startup and shutdown without global variables.


In [ ]:
from contextlib import asynccontextmanager
from fastapi import FastAPI
from fastapi.testclient import TestClient

startup_shutdown_log: list[str] = []

class FakeDbPool:
    def __init__(self):
        startup_shutdown_log.append("pool created")
    def close(self):
        startup_shutdown_log.append("pool closed")

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.db_pool = FakeDbPool()   # startup: runs before 'yield'
    startup_shutdown_log.append("app is now serving requests")
    yield
    app.state.db_pool.close()          # shutdown: runs after 'yield'
    startup_shutdown_log.append("app has stopped serving requests")

app = FastAPI(lifespan=lifespan)

@app.get("/ping")
def ping():
    return {"pool_is_open": True}

with TestClient(app) as client:  # __enter__/__exit__ trigger lifespan startup/shutdown
    print(client.get("/ping").json())

for line in startup_shutdown_log:
    print(line)


In [ ]:
 =>  Using TestClient as a context manager ('with TestClient(app) as client:') is what
       actually triggers the lifespan startup/shutdown -- without 'with', lifespan never
       runs during tests.

 =>  app.state is the standard place to stash things created at startup (a DB pool, an
       HTTP client) so route handlers can access them via 'request.app.state.db_pool'.


In [ ]:
    ############    #############   API Versioning   #############   ##############   

 =>  URL path versioning ('/v1/users', '/v2/users') is the simplest and most common
       approach -- explicit in every request, easy to route, easy to deprecate one version
       while keeping another alive.

 =>  Header-based versioning (e.g. 'Accept: application/vnd.myapi.v2+json') keeps URLs
       stable but is harder to test/debug casually (can't just paste a URL in a browser).

 =>  Whichever you choose, decide it BEFORE your first external client integrates --
       retrofitting versioning onto an unversioned API already in use is painful and often
       requires a breaking migration.


In [ ]:
from fastapi import APIRouter, FastAPI

v1 = APIRouter(prefix="/v1")
v2 = APIRouter(prefix="/v2")

@v1.get("/users/{user_id}")
def get_user_v1(user_id: int):
    return {"id": user_id, "name": "Ada Lovelace"}

@v2.get("/users/{user_id}")
def get_user_v2(user_id: int):
    return {"id": user_id, "full_name": "Ada Lovelace", "active": True}  # renamed + added field

app = FastAPI()
app.include_router(v1)
app.include_router(v2)

from fastapi.testclient import TestClient
client = TestClient(app)
print("v1:", client.get("/v1/users/1").json())
print("v2:", client.get("/v2/users/1").json())


In [ ]:
 =>  v1 clients keep working unmodified (still get 'name') while v2 clients opt in to the
       new shape ('full_name' + 'active') -- both live side by side under one FastAPI app,
       each with its own router.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Add a deprecation warning header (e.g. 'Deprecation: true') to every v1 response,
           and document a sunset date.

 =>  [ ] Extend the lifespan example to create a real async resource (e.g. an httpx
           AsyncClient) at startup and close it at shutdown.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Using '@app.on_event("startup")' in new code -- it's deprecated; lifespan is the
       current pattern.

 =>  Changing a field's meaning or type within the SAME version instead of cutting a new
       version -- this silently breaks every existing client with no way to opt out.
